Code snippet to group VIIRS composite products in a single Dataset

In [ ]:
import xarray as xr
import glob
import os
from datetime import datetime, timedelta
from edelassim.observations import EdelweissGrandesRoussesGrid, reprojection_mf_fsc_l3_to_grid

input_folder = "/home/imperatoren/work/edelweiss_assimilation/data/france/mf_snow_cover/cms"
output_file = "/home/imperatoren/work/edelweiss_assimilation/observations/grandesrousses/meteofrance/snow_cover/composite/mf_fsc_l3_all_grandesrousses_wy_2021_2022.nc"
edelweiss_grandesrousses_grid = EdelweissGrandesRoussesGrid()
viirs_files = glob.glob(f"{input_folder}/*/*/*all.nc")
viirs_grandesrousses_reprojected = []
for f in viirs_files:
    viirs = xr.open_dataset(f, engine="rasterio", mask_and_scale=False)
    reproj = reprojection_mf_fsc_l3_to_grid(
        meteofrance_snow_cover=viirs.data_vars["snow_cover_fraction"], output_grid=edelweiss_grandesrousses_grid
    )
    # t_coord = datetime.strptime(os.path.basename(f)[:8], "%Y%m%d") + timedelta(hours=12)
    # reproj = reproj.expand_dims("time").assign_coords({"time": [t_coord]})
    viirs_grandesrousses_reprojected.append(reproj)


viirs_grandesrousses = xr.concat(viirs_grandesrousses_reprojected, dim="time")
viirs_grandesrousses.to_netcdf(output_file)

In [46]:
import numpy as np
from edelassim.observations import EdelweissGrandesRoussesGrid
from geospatial_grid.georeferencing import georef_netcdf_rioxarray, georef_netcdf_manually
from geospatial_grid.gsgrid import GSGrid
from geospatial_grid.grid_database import PROJ4_MODIS
from geospatial_grid.reprojections import reproject_using_grid
import pyproj
import numpy.ma as ma
import xarray as xr
from rasterio.enums import Resampling

dims = ("y", "x")

NASA_CLASSES = {
    "snow_cover": range(1, 101),
    "no_snow": (0,),
    "clouds": (250,),
    "water": (237, 239),  # inland and ocean
    "no_decision": (201,),
    "night": (211,),
    "missing_data": (251,),
    "L1B_unusable": (252,),
    "bowtie_trim": (253,),
    "L1B_fill": (254,),
    "fill": (255,),
}


def salomonson_appel(ndsi):
    """ "'FRA6T' or 'universal' equation for NDSI calculation.
    See Salomonson, Vincent V., and Igor Appel. "Development of the Aqua MODIS NDSI fractional snow cover algorithm and
      validation results." IEEE Transactions on geoscience and remote sensing 44.7 (2006): 1747-1756."""
    return 1.45 * ndsi - 0.01


def salomonson_appel_regression(ndsi):
    snow_cover_fraction = salomonson_appel(ndsi=ndsi)
    snow_cover_fraction = np.clip(snow_cover_fraction, a_max=1, a_min=0)
    return snow_cover_fraction


def ndsi_snow_cover_to_fraction(
    ndsi_snow_cover_product,
    snow_cover_ndsi_threshold: int = 10,
    max_ndsi: int = 100,
    method: str = "salomonson_appel",
):
    snow_mask = (ndsi_snow_cover_product >= snow_cover_ndsi_threshold) & (ndsi_snow_cover_product <= max_ndsi)
    masked_ndsi_snow_cover = ma.masked_array(ndsi_snow_cover_product, mask=(1 - snow_mask)) / max_ndsi
    if method == "salomonson_appel":
        snow_cover_fraction = salomonson_appel_regression(masked_ndsi_snow_cover)
    # elif method == "mine":
    #     snow_cover_fraction = my_regression(masked_ndsi_snow_cover)
    else:
        raise NotImplementedError(f"Fractional snow cover method {method} not known.")
    out_fractional_snow_cover = (snow_cover_fraction.data * max_ndsi).astype(np.uint8)
    out_fractional_snow_cover = np.where(snow_mask == 1, out_fractional_snow_cover, ndsi_snow_cover_product)

    return out_fractional_snow_cover


def reprojection_l3_nasa_to_grid(nasa_snow_cover: xr.DataArray, output_grid: GSGrid) -> xr.DataArray:

    # Validity "zombie mask": wherever there is at least one non valid pixel, the output grid pixel is set as invalid (<-> cloud)
    # nasa_dataset = nasa_dataset.where(nasa_dataset <= NASA_CLASSES["snow_cover"][-1], NASA_CLASSES["fill"][0])
    resampled_max = reproject_using_grid(
        nasa_snow_cover,
        output_grid=output_grid,
        resampling_method=Resampling.max,
        nodata=NASA_CLASSES["fill"][0],
    )

    resampled_bilinear = reproject_using_grid(
        nasa_snow_cover,
        output_grid=output_grid,
        resampling_method=Resampling.bilinear,
    )

    resampled_nearest = reproject_using_grid(
        nasa_snow_cover,
        output_grid=output_grid,
        resampling_method=Resampling.nearest,
    )

    invalid_mask = resampled_max > NASA_CLASSES["snow_cover"][-1]
    water_mask = resampled_nearest == NASA_CLASSES["water"][0] | NASA_CLASSES["water"][1]
    valid_qualitative_mask = water_mask
    out_snow_cover = resampled_bilinear.where(invalid_mask == False, resampled_max)
    # We readd water resempled with nearest
    out_snow_cover = out_snow_cover.where(valid_qualitative_mask == False, resampled_nearest)

    return out_snow_cover.astype("u1")


folder = "/home/imperatoren/work/edelweiss_assimilation/data/france/viirs_nasa_snow_cover/downloaded"
name = "VNP10A1.A2021300.h18v04.002.2023046165832.h5"
filepath = f"{folder}/{name}"
product_grid_data_variable = xr.open_dataset(filepath, group="HDFEOS/GRIDS/VIIRS_Grid_IMG_2D", engine="netcdf4")
bin_size = xr.open_dataset(filepath, engine="netcdf4").attrs["CharacteristicBinSize"]
nasa_l3_grid = GSGrid(
    resolution=bin_size,
    x0=product_grid_data_variable.coords["XDim"][0].values,
    y0=product_grid_data_variable.coords["YDim"][0].values,
    width=len(product_grid_data_variable.coords["XDim"]),
    height=len(product_grid_data_variable.coords["YDim"]),
)
ndsi_snow_cover = xr.open_dataset(filepath, group="HDFEOS/GRIDS/VIIRS_Grid_IMG_2D/Data Fields", engine="netcdf4").data_vars[
    "NDSI_Snow_Cover"
]

ndsi_snow_cover = ndsi_snow_cover.rename({"XDim": dims[1], "YDim": dims[0]}).assign_coords(
    coords={dims[0]: nasa_l3_grid.ycoords, dims[1]: nasa_l3_grid.xcoords}
)
ndsi_snow_cover = georef_netcdf_rioxarray(data_array=ndsi_snow_cover, crs=pyproj.CRS.from_proj4(PROJ4_MODIS))
ndsi_snow_cover_repr = reprojection_l3_nasa_to_grid(nasa_snow_cover=ndsi_snow_cover, output_grid=EdelweissGrandesRoussesGrid())
fsc = xr.DataArray(
    ndsi_snow_cover_to_fraction(ndsi_snow_cover_repr.values, snow_cover_ndsi_threshold=10, method="salomonson_appel"),
    coords=ndsi_snow_cover_repr.coords,
    dims=ndsi_snow_cover_repr.dims,
)

fsc = georef_netcdf_manually(data_array=fsc, crs=pyproj.CRS.from_epsg(2154))
fsc = fsc.drop_vars("Projection")
fsc.to_netcdf(
    f"/home/imperatoren/work/edelweiss_assimilation/data/grandesrousses/snow_cover/vnp10a1/{name.replace('.h5', '.nc')}"
)
# fsc = xr.open_dataset(f"/home/imperatoren/work/edelweiss_assimilation/data/grandesrousses/snow_cover/vnp10a1/{name}")
# fsc.rio.transform()